In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Setup & Loading Datasets

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ashyou09/apple-global-product-sales-dataset")

print("Path to dataset files:", path)

In [ ]:
df = pd.read_csv("/kaggle/input/apple-global-product-sales-dataset/apple_global_sales_dataset.csv")
df['sale_date'] = pd.to_datetime(df['sale_date'])
df = df.sort_values(['product_name', 'region', 'sale_date'])
print("Data Loaded")

In [ ]:
# Pillar 1: Price Elasticity (The Brand Moat)

In [ ]:
# we use pct_change to see how a % change in price affects % chaange in units
df['price_pct_change'] = df.groupby(['product_name', 'region'])['unit_price_usd'].pct_change()
df['qty_pct_change'] = df.groupby(['product_name', 'region'])['units_sold'].pct_change()

In [ ]:
# Formula: Elasticity = % change in Qty / % change in price
df['elasticity'] = df['qty_pct_change'] / df['price_pct_change']

In [ ]:
# Clean infinites from zero-price-change rows
elasticity_results = df.replace([np.inf, -np.inf], np.nan).dropna(subset=['elasticity'])
brand_moat = elasticity_results.groupby('product_name')['elasticity'].median().abs()

In [ ]:
# Pillar 2: Revenue Concentration (The risk HHI)

In [ ]:
product_revenue = df.groupby('product_name')['revenue_usd'].sum()

In [ ]:
market_share_pct = (product_revenue / product_revenue.sum()) * 100
hhi_score = (market_share_pct**2).sum() 

In [ ]:
# Pillar 3: Sales Stability (The Cash Cow Index)

In [ ]:
# CV = (Standard Deviation Mean / Mean) * 100. Lower = More Stable
stability = df.groupby(['product_name', 'region'])['units_sold'].agg(['std', 'mean'])
stability['cv'] = (stability['std'] / stability['mean']) * 100
stability_index = stability['cv'].groupby('product_name').mean().sort_values()

In [ ]:
# The Executive Summary

In [ ]:
print("\n" + "="*40)
print("APPLE STRATEGIC ANALYSIS SUMMARY")
print("="*40)

print(f"\n[1] REVENUE CONCENTRATION (HHI): {hhi_score:.2f}")
if hhi_score > 2500:
    print("INSIGHT: High Risk. Relies heavily on top-tier products.")
else:
    print("INSIGHT: Diversified. Strong ecosystem balance.")

print("\n[2] BRAND POWER (Elasticity: Lower is Stronger")
print(brand_moat.sort_values().to_string())

print("\n[3] STABILITY INDEX (Top 5 Predictable Products")
print(stability_index.head(5).to_string())

In [ ]:
# Visualization

In [ ]:
plt.figure(figsize=(16,10))
sns.scatterplot(x=brand_moat, y=product_revenue, size=product_revenue, sizes=(100,2500), alpha=0.6, edgecolor='w')
# Adding Labels
for i in range(len(brand_moat)):
    plt.text(x = brand_moat.iloc[i]+0.03, y = product_revenue.iloc[i], s = brand_moat.index[i], fontsize=9, verticalalignment='center', ha='left',
            bbox=dict(facecolor='white', alpha=0.5, edgecolor='none', pad=1))
plt.title("Apple Portfolio: Pricing Power vs Revenue", fontsize = 18, pad=20)
plt.title("Price Elasticity (customer sensitivity)", fontsize = 12)
plt.title("Total Revenue (USD)", fontsize = 12)
plt.xlim(brand_moat.min() - 0.01, brand_moat.max() + 0.05)
plt.grid(True, alpha=0.8, linestyle=':')
plt.tight_layout()             
plt.show()